# BiTro ST Pipeline (Notebook)

This notebook provides a configurable start-to-end workflow for spatial transcriptomics in BiTro.
Edit the config cell first, then run cells step by step.

In [ ]:
import os
import subprocess
from pathlib import Path

def run_cmd(cmd, env=None, cwd=None):
    print(f"\n[RUN] {' '.join(cmd)}")
    if env is not None:
        debug_keys = [
            'OUTPUT_DIR', 'HEST_DATA_DIR', 'SPATIAL_FEATURE_DIR', 'SPATIAL_GRAPH_DIR',
            'GENE_FILE', 'SAMPLE_IDS', 'CV_MODE', 'CV_HELDOUT', 'CUDA_DEVICE_ID',
            'NUMBA_CACHE_DIR', 'USE_TRANSFER_LEARNING', 'FREEZE_BACKBONE'
        ]
        print('[ENV]')
        for key in debug_keys:
            if key in env:
                print(f'  {key}={env[key]}')
    result = subprocess.run(cmd, env=env, cwd=cwd, text=True, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


In [ ]:
# =========================
# Editable configuration
# =========================
PROJECT_ROOT = Path('.').resolve()

HEST_DATA_DIR = Path('/data/yujk/hovernet2feature/BiTro/demo_data/HEST')
SPATIAL_FEATURE_DIR = Path('/data/yujk/hovernet2feature/BiTro/demo_data/Feature/Spatial')
SPATIAL_GRAPH_DIR = Path('/data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Spatial')
DINOV3_WEIGHTS = Path('/data/yujk/hovernet2feature/BiTro/demo_data/weights/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth')
GENE_FILE = Path('/data/yujk/hovernet2feature/BiTro/demo_data/Gene/her_hvg_cut_1000.txt')

# Set SAMPLE_IDS = None to process all discovered samples
SAMPLE_IDS = ['SPA148', 'SPA147']

# Feature extraction settings
DINO_BATCH_SIZE = 256
CELL_BATCH_SIZE = 50000
NUM_WORKERS = 8
N_CLUSTERS = 7
INTER_SPOT_K = 6

# Training settings
TRAIN_OUTPUT_DIR = PROJECT_ROOT / 'log_normalized_notebook'
CV_MODE = 'loo'  # 'kfold' or 'loo'
CV_HELDOUT = None  # e.g. 'TENX147' when CV_MODE='loo'
USE_TRANSFER_LEARNING = False
FREEZE_BACKBONE = False

# NOTE: transfer learning weights can be overridden with BULK_MODEL_PATH in the training env.
# The default path is still defined inside spitial_model/train.py.

print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
required_paths = [
    HEST_DATA_DIR,
    DINOV3_WEIGHTS,
    GENE_FILE,
    PROJECT_ROOT / 'utils' / 'extract_spatial_features_dinov3.py',
    PROJECT_ROOT / 'utils' / 'cluster_kmeans_features.py',
    PROJECT_ROOT / 'spitial_model' / 'train.py',
]
for p in required_paths:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")

st_dir = HEST_DATA_DIR / 'st'
all_samples = sorted([x.stem for x in st_dir.glob('*.h5ad')]) if st_dir.exists() else []
selected_samples = all_samples if SAMPLE_IDS is None else [s for s in SAMPLE_IDS if s in all_samples]
print(f'All samples: {len(all_samples)}')
print(f'Selected samples: {len(selected_samples)}')
print(selected_samples[:20])

In [ ]:
# Step 1: extract per-cell DINOv3 features (with per-sample PCA)
import torch
from utils.extract_spatial_features_dinov3 import (
    HESTCellFeatureExtractor,
    get_expected_num_cells,
    is_sample_completed,
)

SPATIAL_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

extractor = HESTCellFeatureExtractor(
    hest_data_dir=str(HEST_DATA_DIR),
    output_dir=str(SPATIAL_FEATURE_DIR),
    dinov3_model_path=str(DINOV3_WEIGHTS),
    bulk_pca_model_path=None,
    device=device,
    dino_batch_size=DINO_BATCH_SIZE,
    cell_batch_size=CELL_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    assign_spot=True,
)

for sid in selected_samples:
    expected = get_expected_num_cells(str(HEST_DATA_DIR), sid)
    if is_sample_completed(str(SPATIAL_FEATURE_DIR), sid, expected):
        print(f'[SKIP] {sid} already completed')
        continue
    print(f'\n[PROCESS] {sid}')
    _ = extractor.process_sample_with_independent_pca(sid)

In [ ]:
# Step 2: add KMeans cluster labels into each .npz feature file
from utils.cluster_kmeans_features import process_one_file

for sid in selected_samples:
    npz_path = SPATIAL_FEATURE_DIR / f'{sid}_combined_features.npz'
    if not npz_path.exists():
        print(f'[MISS] {npz_path}')
        continue
    process_one_file(str(npz_path), n_clusters=N_CLUSTERS, backup=True)

In [ ]:
# Step 3: construct spatial graphs
from utils.spatial_graph_construction import HESTDirectReader

SPATIAL_GRAPH_DIR.mkdir(parents=True, exist_ok=True)
builder = HESTDirectReader(
    hest_data_dir=str(HEST_DATA_DIR),
    sample_ids=selected_samples,
    features_dir=str(SPATIAL_FEATURE_DIR),
    inter_spot_k_neighbors=INTER_SPOT_K,
)
builder.load_sample_data()
builder.process_all_samples()
metadata = builder.save_graphs(str(SPATIAL_GRAPH_DIR))
print('Graph samples:', len(metadata))

In [ ]:
# Step 4: train spatial model
env = os.environ.copy()
env['NUMBA_CACHE_DIR'] = '/tmp/numba_cache'

train_cmd = [
    'python', 'spitial_model/train.py',
    '--output_dir', str(TRAIN_OUTPUT_DIR),
    '--hest_data_dir', str(HEST_DATA_DIR),
    '--features_dir', str(SPATIAL_FEATURE_DIR),
    '--graph_dir', str(SPATIAL_GRAPH_DIR),
    '--gene_file', str(GENE_FILE),
    '--sample_ids', ','.join(selected_samples),
    '--cv_mode', CV_MODE,
    '--cuda_device_id', '0',
    '--use_transfer_learning', str(USE_TRANSFER_LEARNING).lower(),
    '--freeze_backbone', str(FREEZE_BACKBONE).lower(),
    '--numba_cache_dir', env['NUMBA_CACHE_DIR'],
]
if CV_HELDOUT:
    train_cmd.extend(['--cv_heldout', CV_HELDOUT])

print('Step 4 configuration:')
print('  HEST_DATA_DIR =', HEST_DATA_DIR)
print('  SPATIAL_FEATURE_DIR =', SPATIAL_FEATURE_DIR)
print('  SPATIAL_GRAPH_DIR =', SPATIAL_GRAPH_DIR)
print('  selected_samples =', selected_samples)
print('  CV_MODE =', CV_MODE)
print('  CV_HELDOUT =', CV_HELDOUT)

run_cmd(train_cmd, env=env, cwd=str(PROJECT_ROOT))

In [ ]:
# Step 5: quick artifact check
print('Spatial feature dir exists:', SPATIAL_FEATURE_DIR.exists())
print('Spatial graph dir exists:', SPATIAL_GRAPH_DIR.exists())
print('Train output dir exists:', TRAIN_OUTPUT_DIR.exists())

if SPATIAL_GRAPH_DIR.exists():
    for p in sorted(SPATIAL_GRAPH_DIR.glob('*')):
        print(' -', p.name)